# Carga del Modelo EfficientNet a Hugging Face Hub

Este notebook sube el modelo entrenado (EfficientNet) a Hugging Face Hub,
permitiendo que sea accesible desde la aplicación Streamlit sin necesidad
de almacenar archivos locales.

**Flujo:**
1. Autenticarse en Hugging Face
2. Crear repositorio del modelo
3. Subir archivo del modelo (.keras)
4. Subir metadatos (clases, accuracies, thresholds)

In [15]:
# Configuración e importaciones
from huggingface_hub import HfApi, login
import os
from dotenv import load_dotenv

# Cargar token de HF desde variables de entorno
# PASOS para crear tu token:
#   1. Ir a huggingface.co → Settings → Access Tokens
#   2. Crear nuevo token con rol 'write'
#   3. Guardar en variable de entorno HF_TOKEN (archivo .env o export)
load_dotenv()  # Carga desde .env si existe

HF_TOKEN = os.getenv("HF_TOKEN")
if not HF_TOKEN:
    raise ValueError("❌ Variable HF_TOKEN no encontrada. Configúrala en .env")

login(token=HF_TOKEN)
api = HfApi()

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [16]:
# Crear repositorio en Hugging Face
# Formato REPO_ID: "usuario/nombre-repo"
REPO_ID = "IrvingRD/Sideral"

# Crear o validar que existe el repositorio
# exist_ok=True evita error si ya existe
# private=False permite que Streamlit Cloud lo acceda
api.create_repo(
    repo_id=REPO_ID,
    repo_type="model",
    exist_ok=True,           # No fallar si ya existe
    private=False            # Público para Streamlit Cloud
)
print(f"✓ Repositorio disponible: https://huggingface.co/{REPO_ID}")

✓ Repositorio creado: https://huggingface.co/IrvingRD/Sideral


In [17]:
# Definir clases en orden fijo
# IMPORTANTE: Este orden debe coincidir exactamente con el del modelo entrenado
# (no puede reordenarse, causa predicciones incorrectas)
CLASSES = ['elliptical', 'edge_on', 'merger', 'spiral']
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}

In [19]:
# Subir el modelo entrenado a Hugging Face
# Archivo: model.keras contiene pesos y arquitectura de EfficientNetB0
api.upload_file(
    path_or_fileobj="../data/models/efficientnet_gz2_fase2_best.keras",
    path_in_repo="efficientnet_gz2_fase2_best.keras",
    repo_id=REPO_ID,
    repo_type="model"
)
print("✓ Modelo subido correctamente a Hugging Face")

# Preparar metadatos del modelo para documentación y reproducibilidad
# Estos datos son críticos para la app Streamlit:
#   - classes: etiquetas de clasificación (4 clases morfológicas)
#   - img_size: tamaño de entrada (224x224 para EfficientNetB0)
#   - thresholds: valores de confianza usados en entrenamiento
#   - métricas: resultados en conjunto de validación y test
import json

metadata = {
    "classes": CLASSES,
    "class_to_idx": CLASS_TO_IDX,
    "img_size": 224,                       # Entrada EfficientNetB0
    "threshold_main": 0.9,                 # Umbral para clasificaciones principales
    "threshold_merger": 0.7,               # Umbral más bajo para detectar mergers
    "val_accuracy": 0.9144,                # Accuracy en validación
    "test_recall": {
        "elliptical": 0.916,               # Recall por clase (test set)
        "edge_on": 0.984,
        "merger": 0.800,
        "spiral": 0.899
    }
}

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

✓ Modelo subido correctamente


In [20]:
# Guardar y subir metadatos del modelo
# El archivo JSON contiene información de clases y métricas
# necesaria por Streamlit para validar y usar el modelo

with open("../data/models/model_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

api.upload_file(
    path_or_fileobj="../data/models/model_metadata.json",
    path_in_repo="model_metadata.json",
    repo_id=REPO_ID,
    repo_type="model"
)

print("✓ Metadatos subidos correctamente")
print(f"\n  ✓ Modelo disponible en: https://huggingface.co/{REPO_ID}")
print(f"  ✓ Listo para usar en Streamlit Cloud")

✓ Metadata subida correctamente

  URL del modelo: https://huggingface.co/IrvingRD/Sideral
